# Verbatim, time-aligned transcripts: CrisperWhisper 2.0 + MFA v3

Use a GPU runtime (**Runtime -> Change runtime type -> GPU**). The code is cloned to
`/content` (fast local disk). Recordings are read from Drive, and every result is written back to Drive:

```
DATA_DIR/audio/LCC_Data/<participant>/...   recordings (same layout as the local project)
DATA_DIR/data/                              wav, transcripts, MFA corpus/output, aligned results
DATA_DIR/cache/ct2/                         converted CrisperWhisper model (reused next session)
```

Every step skips work already on Drive, so after a disconnect just re-run the cells.

## 0. Setup

In [ ]:
REPO_URL = "https://github.com/<you>/cognitive_load.git"   # <- your repo
DATA_DIR = "/content/drive/MyDrive/LCC_Data"         # <- folder containing audio/LCC_Data

from google.colab import drive
drive.mount("/content/drive")

import os
if not os.path.exists("/content/cognitive_load"):
    !git clone -q {REPO_URL} /content/cognitive_load
%cd /content/cognitive_load
!git pull -q
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv

In [ ]:
# CrisperWhisper weights are under a non-commercial research licence. If the HF repo asks
# you to accept terms, do so on the model page and add an HF_TOKEN Colab secret.
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    print("no HF_TOKEN secret set (fine if the model is not gated)")

## 1. Convert audio, smoke-test CrisperWhisper on one file
Check the printed IDs (participant / day / prompt / condition are parsed from file names)
and read the one transcript before running everything.

In [ ]:
!CW_ARGS="--limit 1" bash run_pipeline.sh {DATA_DIR} 1 2
!head -c 1500 {DATA_DIR}/data/transcripts/crisperwhisper/*.txt

## 2. Run everything else
`CW_WORKERS` = number of model copies on the GPU (~4 GB each in fp16): 2 on a 16 GB T4 (3 if nvidia-smi shows room),
4-5 on a 24 GB L4. The first MFA run installs MFA from conda-forge (a few minutes).

In [ ]:
!CW_WORKERS=2 bash run_pipeline.sh {DATA_DIR}

## 3. Look at the results

In [ ]:
import pandas as pd
summary = pd.read_csv(f"{DATA_DIR}/data/aligned/summary.csv")
display(summary.sort_values("mfa_coverage").head(10))
display(summary.groupby("condition")[["wpm", "fillers_per_min", "n_pauses", "mean_pause_s"]].mean())

Re-run single steps with e.g. `!bash run_pipeline.sh {DATA_DIR} 3 4 5`
(say after changing chunking or pause settings: `!MIN_PAUSE=0.2 bash run_pipeline.sh {DATA_DIR} 5`).